In [1]:
import os, sys, json, glob, random, math, time, zipfile
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = "/kaggle/working"
SUB_DIR = f"{WORK}/p2_submissions"; CK2 = f"{WORK}/p2_ckpt"
os.makedirs(SUB_DIR, exist_ok=True); os.makedirs(CK2, exist_ok=True)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

cuda Tesla T4


In [2]:
import os, glob

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pt", ".json")) and f != "prediction.json":
            p = os.path.join(root, f)
            print(f"{os.path.getsize(p)/1e6:9.1f} MB  {p}")

def pick_ckpt(name):
    hits = [p for p in glob.glob("/kaggle/input/**/*", recursive=True)
            if os.path.isfile(p) and p.endswith(".pt") and name in os.path.basename(p)
            and "_last" not in os.path.basename(p)]
    return max(hits, key=os.path.getsize) if hits else None

CKPT = {n: pick_ckpt(n) for n in ["A_decoder", "B_encdec", "C_hybrid", "D_efficient"]}
tok_hits = glob.glob("/kaggle/input/**/*bpe32k*.json", recursive=True)
TOK_PATH = tok_hits[0] if tok_hits else None

print("\nCKPT:", CKPT)
print("TOK_PATH:", TOK_PATH)
assert all(CKPT.values()), f"missing: {[k for k, v in CKPT.items() if not v]}"
assert TOK_PATH, "tokenizer not found"

    296.6 MB  /kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__B_encdec.pt
    268.2 MB  /kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__A_decoder.pt
    277.7 MB  /kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__C_hybrid.pt
    221.1 MB  /kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__D_efficient.pt
      2.3 MB  /kaggle/input/datasets/samswe/bpefile/bpe32k.json

CKPT: {'A_decoder': '/kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__A_decoder.pt', 'B_encdec': '/kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__B_encdec.pt', 'C_hybrid': '/kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__C_hybrid.pt', 'D_efficient': '/kaggle/input/datasets/samswe/actualphase1result/__kaggle__working__ckpt__D_efficient.pt'}
TOK_PATH: /kaggle/input/datasets/samswe/bpefile/bpe32k.json


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [4]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

os.makedirs("/kaggle/temp/hf_p2", exist_ok=True)
local = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                          token=UserSecretsClient().get_secret("HF_TOKEN"),
                          allow_patterns=["Phase_2_LNER/*"], local_dir="/kaggle/temp/hf_p2")
ROOT = f"{local}/Phase_2_LNER"
for r, d, f in os.walk(ROOT): print(r, f)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/kaggle/temp/hf_p2/Phase_2_LNER []
/kaggle/temp/hf_p2/Phase_2_LNER/Train ['labels.txt', 'fold2-text.json', 'fold1-gold.json', 'fold2-gold.json', 'fold1-text.json']
/kaggle/temp/hf_p2/Phase_2_LNER/Test ['fold3-text.json']


In [5]:
def jload(p): return json.load(open(p, encoding="utf-8"))

text1, text2 = jload(f"{ROOT}/Train/fold1-text.json"), jload(f"{ROOT}/Train/fold2-text.json")
gold1, gold2 = jload(f"{ROOT}/Train/fold1-gold.json"), jload(f"{ROOT}/Train/fold2-gold.json")
test_text = jload(f"{ROOT}/Test/fold3-text.json")
LABELS = [l.strip() for l in open(f"{ROOT}/Train/labels.txt") if l.strip()]
print(LABELS, "| train docs:", len(text1) + len(text2), "| test docs:", len(test_text))

k = next(iter(gold1)); print("gold sample:", k, type(gold1[k]), str(gold1[k])[:300])

def ent_tuple(x):
    if isinstance(x, (list, tuple)): return (int(x[0]), int(x[1]), x[2])
    g = lambda *ks: next(x[k] for k in ks if k in x)
    return (int(g("start", "start_char", "start_offset")), int(g("end", "end_char", "end_offset")), g("label", "type", "entity"))

def norm_gold(g):
    return {d: [ent_tuple(x) for x in ents] for d, ents in g.items()}

texts = {**text1, **text2}
gold = {**norm_gold(gold1), **norm_gold(gold2)}
d = next(iter(gold)); s, e, l = gold[d][0]; print("check:", l, repr(texts[d][s:e]))

ids_all = sorted(texts); random.Random(SEED).shuffle(ids_all)
nval = max(1, int(0.1 * len(ids_all)))
val_ids, train_ids = ids_all[:nval], ids_all[nval:]
print("train/val docs:", len(train_ids), len(val_ids))

tags = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
T2I = {t: i for i, t in enumerate(tags)}; I2T = {i: t for t, i in T2I.items()}
NTAGS = len(tags)

['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO'] | train docs: 70 | test docs: 35
gold sample: 115651329 <class 'list'> [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana'], [359, 369, 'COURT', 'High Court'], [575, 585, 'DATE', '18.11.1994'], [637, 644, 'WIT', 'Jagdish'], [695, 704, 'WIT', 'Mange Ram'], [792, 800, 'APP', 'Jage Ram'], [1122, 
check: RESP 'STATE OF HARYANA'
train/val docs: 63 7


In [6]:
VOCAB_SIZE, PAD_ID = 32000, 0

class _Ctx: kmask = None          # key-padding mask used by bidirectional attention
CTX = _Ctx()

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rotary_freqs(dim, max_seq, base=10000.0, device="cpu"):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    f = torch.outer(torch.arange(max_seq, device=device).float(), inv)
    return torch.cos(f), torch.sin(f)

def apply_rotary(x, cos, sin):
    d = x.shape[-1]; x1, x2 = x[..., :d // 2], x[..., d // 2:]
    cos = cos[:x.shape[-2], :d // 2][None, None]; sin = sin[:x.shape[-2], :d // 2][None, None]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1)

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False); self.w2 = nn.Linear(dim, hidden, bias=False); self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x): return self.w3(F.silu(self.w1(x)) * self.w2(x))

# ---- Model A
class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.n_heads, self.hd, self.causal = n_heads, dim // n_heads, True
        self.qkv = nn.Linear(dim, 3 * dim, bias=False); self.proj = nn.Linear(dim, dim, bias=False)
        cos, sin = rotary_freqs(self.hd, max_seq)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.hd).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        if self.causal: out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:           out = F.scaled_dot_product_attention(q, k, v, attn_mask=CTX.kmask)
        return self.proj(out.transpose(1, 2).reshape(B, T, C))

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq, mlp_mult=4):
        super().__init__()
        self.norm1 = RMSNorm(dim); self.attn = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim); self.mlp = SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); return x + self.mlp(self.norm2(x))

class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_layers=12, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([DecoderBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f = RMSNorm(dim); self.head = nn.Linear(dim, vocab, bias=False); self.head.weight = self.tok_emb.weight

# ---- generic attention (B, C, D)
class Attn(nn.Module):
    def __init__(self, dim, n_heads, max_seq, n_kv_heads=None, causal=True, window=None, rope=True):
        super().__init__()
        self.h, self.kv = n_heads, (n_kv_heads or n_heads); self.hd = dim // n_heads
        self.causal, self.window, self.rope = causal, window, rope
        self.q = nn.Linear(dim, self.h * self.hd, bias=False); self.k = nn.Linear(dim, self.kv * self.hd, bias=False)
        self.v = nn.Linear(dim, self.kv * self.hd, bias=False); self.proj = nn.Linear(self.h * self.hd, dim, bias=False)
        if rope:
            cos, sin = rotary_freqs(self.hd, max_seq)
            self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
    def _mask(self, T, device):
        i = torch.arange(T, device=device)[:, None]; j = torch.arange(T, device=device)[None, :]
        return (j <= i) & (i - j < self.window)
    def forward(self, x, ctx=None):
        B, T, C = x.shape; src = x if ctx is None else ctx; S = src.size(1)
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        v = self.v(src).view(B, S, self.kv, self.hd).transpose(1, 2)
        if self.rope and ctx is None: q, k = apply_rotary(q, self.cos, self.sin), apply_rotary(k, self.cos, self.sin)
        if self.kv != self.h:
            r = self.h // self.kv; k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
        if ctx is None and self.causal and self.window:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=self._mask(T, x.device))
        elif ctx is None and not self.causal:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=CTX.kmask)
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=(self.causal and ctx is None))
        return self.proj(out.transpose(1, 2).reshape(B, T, self.h * self.hd))

# ---- Model B
class EncBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.attn = RMSNorm(dim), Attn(dim, nh, ms, causal=False)
        self.n2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.attn(self.n1(x)); return x + self.mlp(self.n2(x))

class CrossDecBlock(nn.Module):
    def __init__(self, dim, nh, ms, mlp_mult=4):
        super().__init__()
        self.n1, self.self_attn = RMSNorm(dim), Attn(dim, nh, ms, causal=True)
        self.n2, self.cross = RMSNorm(dim), Attn(dim, nh, ms, causal=False, rope=False)
        self.n3, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))

class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab=VOCAB_SIZE, dim=768, n_enc=6, n_dec=6, n_heads=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.enc = nn.ModuleList([EncBlock(dim, n_heads, max_seq) for _ in range(n_enc)])
        self.dec = nn.ModuleList([CrossDecBlock(dim, n_heads, max_seq) for _ in range(n_dec)])
        self.norm_e, self.norm_f = RMSNorm(dim), RMSNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False); self.head.weight = self.tok_emb.weight

# ---- Models C, D
class GatedLinearRecurrence(nn.Module):
    def __init__(self, dim, chunk=64):
        super().__init__()
        self.inp = nn.Linear(dim, 4 * dim, bias=False)
        self.conv = nn.Conv1d(dim, dim, 4, groups=dim, padding=3)
        self.out = nn.Linear(dim, dim, bias=False); self.chunk = chunk
    def _scan(self, log_a, z):
        B, T, C = z.shape; L = self.chunk; pad = (-T) % L
        if pad: log_a, z = F.pad(log_a, (0, 0, 0, pad)), F.pad(z, (0, 0, 0, pad))
        n = (T + pad) // L
        log_a, z = log_a.reshape(B, n, L, C), z.reshape(B, n, L, C)
        A = log_a.cumsum(2); local = torch.exp(A) * torch.cumsum(torch.exp(-A) * z, 2)
        prev, outs = z.new_zeros(B, C), []
        for j in range(n):
            o = local[:, j] + torch.exp(A[:, j]) * prev[:, None]; prev = o[:, -1]; outs.append(o)
        return torch.stack(outs, 1).reshape(B, n * L, C)[:, :T]
    def forward(self, x):
        B, T, C = x.shape
        u, ga, gi, go = self.inp(x).chunk(4, -1)
        u = self.conv(u.transpose(1, 2))[..., :T].transpose(1, 2)
        log_a = -0.05 * torch.sigmoid(ga.float()); z = (torch.sigmoid(gi) * F.silu(u)).float()
        return self.out(self._scan(log_a, z).to(x.dtype) * F.silu(go))

class Block(nn.Module):
    def __init__(self, dim, mixer, mlp_mult=4):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp = RMSNorm(dim), SwiGLU(dim, int(dim * mlp_mult * 2 / 3))
    def forward(self, x):
        x = x + self.mixer(self.norm1(x)); return x + self.mlp(self.norm2(x))

class GenericLM(nn.Module):
    def __init__(self, make_block, vocab=VOCAB_SIZE, dim=768, n_layers=12, max_seq=2048):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([make_block(i) for i in range(n_layers)])
        self.norm_f = RMSNorm(dim); self.head = nn.Linear(dim, vocab, bias=False); self.head.weight = self.tok_emb.weight

def make_hybrid(dim=768, n_heads=12, max_seq=2048, attn_every=3):
    def f(i):
        mixer = Attn(dim, n_heads, max_seq) if (i + 1) % attn_every == 0 else GatedLinearRecurrence(dim)
        return Block(dim, mixer)
    return f

def make_efficient(dim=768, n_heads=12, n_kv=4, window=256, global_every=4, max_seq=2048):
    def f(i):
        is_global = (i + 1) % global_every == 0
        return Block(dim, Attn(dim, n_heads, max_seq, n_kv_heads=n_kv, window=None if is_global else window), mlp_mult=3)
    return f

def build_p1(name):
    return {"A_decoder": lambda: DecoderOnlyLM(), "B_encdec": lambda: EncoderDecoderLM(),
            "C_hybrid": lambda: GenericLM(make_hybrid()), "D_efficient": lambda: GenericLM(make_efficient())}[name]()

class Backbone(nn.Module):
    """Hidden states of a Phase-1 model (LM head dropped; for B only the encoder is kept)."""
    def __init__(self, lm, kind):
        super().__init__()
        self.tok_emb = lm.tok_emb
        self.blocks = lm.enc if kind == "B_encdec" else lm.blocks
        self.norm = lm.norm_e if kind == "B_encdec" else lm.norm_f
    def forward(self, ids):
        x = self.tok_emb(ids)
        for b in self.blocks: x = b(x)
        return self.norm(x)

def load_phase1(name, bidir=False, keep=None):
    lm = build_p1(name)
    lm.load_state_dict(torch.load(CKPT[name], map_location="cpu"), strict=True)   # Phase-1 weights
    bb = Backbone(lm, name)
    if keep is not None:                                    # layer-dropped student
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep])
    if bidir:                                               # fine-tune with full (bidirectional) attention
        for m in bb.modules():
            if isinstance(m, (CausalSelfAttention, Attn)): m.causal = False
    return bb

for n in CKPT:
    bb = load_phase1(n); print(n, f"backbone {sum(p.numel() for p in bb.parameters())/1e6:.1f}M params"); del bb

A_decoder backbone 109.5M params
B_encdec backbone 67.1M params
C_hybrid backbone 114.3M params
D_efficient backbone 85.9M params


In [9]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOK_PATH)

def enc_many(dct):
    keys = list(dct)
    es = tok.encode_batch([dct[k] for k in keys], add_special_tokens=False)
    return {k: (list(e.ids), list(e.offsets)) for k, e in zip(keys, es)}

ENC = enc_many({d: texts[d] for d in ids_all})
ENC_TEST = enc_many(test_text)
lens = np.array([len(v[0]) for v in ENC.values()])
print("tokens/doc  min/median/p90/max:", lens.min(), int(np.median(lens)), int(np.percentile(lens, 90)), lens.max())

def char_to_tags(ents, offsets):
    ends = np.array([b for a, b in offsets]); n = len(offsets); out = [0] * n
    for s, e, l in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        i = int(np.searchsorted(ends, s, side="right")); first = True
        while i < n and offsets[i][0] < e:
            a, b = offsets[i]
            if b > a and out[i] == 0:
                out[i] = T2I[("B-" if first else "I-") + l]; first = False
            i += 1
    return out

def tags_to_spans(tag_ids, offsets, text):
    spans, cur = [], None
    def close():
        nonlocal cur
        if cur:
            s, e, l = cur
            while s < e and text[s].isspace(): s += 1
            while e > s and text[e - 1].isspace(): e -= 1
            if e > s: spans.append((s, e, l))
        cur = None
    for t, (a, b) in zip(tag_ids, offsets):
        if b <= a: continue
        if t == 0: close(); continue
        pre, l = I2T[t].split("-", 1)
        if pre == "B" or cur is None or cur[2] != l: close(); cur = [a, b, l]
        else: cur[1] = b
    close(); return spans

def strict_macro(pred, gold_d, doc_ids):
    P, R, F1 = [], [], []
    for lab in LABELS:
        pr = {(d, s, e) for d in doc_ids for s, e, l in pred.get(d, []) if l == lab}
        gt = {(d, s, e) for d in doc_ids for s, e, l in gold_d.get(d, []) if l == lab}
        tp = len(pr & gt); p = tp / len(pr) if pr else 0.0; r = tp / len(gt) if gt else 0.0
        P.append(p); R.append(r); F1.append(2 * p * r / (p + r) if p + r else 0.0)
    return dict(P=float(np.mean(P)), R=float(np.mean(R)), F1=float(np.mean(F1)), per_label=dict(zip(LABELS, np.round(F1, 3))))

def error_breakdown(pred, gold_d, doc_ids):
    c = dict(exact=0, boundary=0, label_confusion=0, false_pos=0, missed=0)
    for d in doc_ids:
        P, G = set(pred.get(d, [])), set(gold_d.get(d, []))
        for g in G:
            if g in P: c["exact"] += 1; continue
            ov = [p for p in P if p[0] < g[1] and g[0] < p[1]]
            if any(p[2] == g[2] for p in ov): c["boundary"] += 1
            elif ov: c["label_confusion"] += 1
            else: c["missed"] += 1
        c["false_pos"] += sum(1 for p in P if p not in G and not any(p[0] < g[1] and g[0] < p[1] for g in G))
    return c

tokens/doc  min/median/p90/max: 2124 5561 9988 23842


In [10]:
MAXLEN, OVERLAP = 512, 128
NEG_KEEP = 0.3     # keep only 30% of windows that contain no entity (boilerplate / imbalance control)

def windows(n, L=MAXLEN, ov=OVERLAP):
    if n <= L: return [(0, n)]
    step, out, i = L - ov, [], 0
    while i + L < n: out.append((i, i + L)); i += step
    out.append((n - L, n)); return out

def build_examples(doc_ids, ENCD, labeled=True, neg_keep=1.0, seed=0):
    rng, ex = random.Random(seed), []
    for d in doc_ids:
        ids_, offs = ENCD[d]
        lab = char_to_tags(gold[d], offs) if labeled else [0] * len(ids_)
        for a, b in windows(len(ids_)):
            y = lab[a:b]
            if labeled and neg_keep < 1.0 and not any(y) and rng.random() > neg_keep: continue
            ex.append(dict(doc=d, a=a, ids=ids_[a:b], y=y))
    return ex

TR_EX = build_examples(train_ids, ENC, True, NEG_KEEP)
VA_EX = build_examples(val_ids, ENC, True, 1.0)
TE_EX = build_examples(sorted(test_text), ENC_TEST, False)
print("train/val/test windows:", len(TR_EX), len(VA_EX), len(TE_EX))

def collate(batch):
    L = max(len(x["ids"]) for x in batch)
    ids = torch.full((len(batch), L), PAD_ID); y = torch.full((len(batch), L), -100); m = torch.zeros(len(batch), L, dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["ids"]); ids[i, :n] = torch.tensor(x["ids"]); y[i, :n] = torch.tensor(x["y"]); m[i, :n] = 1
    return ids, m, y

train/val/test windows: 1069 108 648


In [11]:
oracle = {d: tags_to_spans(char_to_tags(gold[d], ENC[d][1]), ENC[d][1], texts[d]) for d in val_ids}
print("ORACLE strict F1:", strict_macro(oracle, gold, val_ids))

ORACLE strict F1: {'P': 0.9052862548760294, 'R': 0.9052862548760294, 'F1': 0.9052862548760294, 'per_label': {'APP': np.float64(0.985), 'RESP': np.float64(1.0), 'A.COUNSEL': np.float64(1.0), 'R.COUNSEL': np.float64(1.0), 'JUDGE': np.float64(1.0), 'WIT': np.float64(0.0), 'AUTH': np.float64(1.0), 'COURT': np.float64(1.0), 'STAT': np.float64(0.948), 'PREC': np.float64(0.93), 'DATE': np.float64(1.0), 'CASENO': np.float64(1.0)}}


In [12]:
class NERModel(nn.Module):
    def __init__(self, bb, head="linear", hid=768):
        super().__init__(); self.bb = bb; self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(hid, NTAGS) if head == "linear" else nn.Sequential(
            nn.Linear(hid, hid), nn.GELU(), nn.Dropout(0.1), nn.Linear(hid, NTAGS))
    def forward(self, ids, mask):
        CTX.kmask = mask[:, None, None, :].bool()
        try: h = self.bb(ids)
        finally: CTX.kmask = None
        return self.head(self.drop(h))

class LoRALinear(nn.Module):                 # W' = W + s*B A (additive)
    def __init__(self, base, r=8, alpha=16):
        super().__init__(); self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + (x @ self.A.T @ self.B.T) * self.s

class LoRMALinear(nn.Module):                # W' = (I + B A) W (multiplicative)
    def __init__(self, base, r=8):
        super().__init__(); self.base = base
        self.A = nn.Parameter(torch.randn(r, base.out_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        W = self.base.weight
        return F.linear(x, W + self.B @ (self.A @ W), self.base.bias)

def inject(root, cls, targets, **kw):
    for mod in list(root.modules()):
        for cname, child in list(mod.named_children()):
            if cname in targets and isinstance(child, nn.Linear): setattr(mod, cname, cls(child, **kw))

def set_mode(model, mode, targets=(), r=8):
    """full | lora | lorma | partial (last 4 blocks) | norm (RMSNorm scales = BitFit analog; Phase-1 has no biases)"""
    if mode == "full": return model
    for p in model.bb.parameters(): p.requires_grad = False
    if mode == "lora":  inject(model.bb, LoRALinear, targets, r=r)
    if mode == "lorma": inject(model.bb, LoRMALinear, targets, r=r)
    if mode == "partial":
        for b in list(model.bb.blocks)[-4:]:
            for p in b.parameters(): p.requires_grad = True
        for p in model.bb.norm.parameters(): p.requires_grad = True
    if mode == "norm":
        for m in model.bb.modules():
            if isinstance(m, RMSNorm):
                for p in m.parameters(): p.requires_grad = True
    for p in model.head.parameters(): p.requires_grad = True
    return model

def count(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad); tot = sum(p.numel() for p in model.parameters())
    return f"trainable {tr/1e6:.2f}M / total {tot/1e6:.2f}M"

In [13]:
RESULTS = []
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda")

def class_weights(examples, cap=20.0):
    c = np.ones(NTAGS)
    for x in examples:
        y = np.array(x["y"]); c += np.bincount(y[y >= 0], minlength=NTAGS)
    w = (c.sum() / c) ** 0.5; w = w / w[0]; return torch.tensor(np.minimum(w, cap), dtype=torch.float)

def predict_probs(model, examples, ENCD, bs=16):
    model.eval()
    acc = {d: (np.zeros((len(ENCD[d][0]), NTAGS), np.float32), np.zeros(len(ENCD[d][0]), np.float32)) for d in {x["doc"] for x in examples}}
    order = sorted(range(len(examples)), key=lambda i: len(examples[i]["ids"]))
    with torch.no_grad(), torch.autocast(**AMP):
        for i in range(0, len(order), bs):
            b = [examples[j] for j in order[i:i + bs]]; ids, m, _ = collate(b)
            p = model(ids.to(DEVICE), m.to(DEVICE)).float().softmax(-1).cpu().numpy()
            for k, x in enumerate(b):
                n = len(x["ids"]); S, C = acc[x["doc"]]; S[x["a"]:x["a"] + n] += p[k, :n]; C[x["a"]:x["a"] + n] += 1
    return {d: S / np.maximum(C, 1)[:, None] for d, (S, C) in acc.items()}

def probs_to_spans(probs, ENCD, text_d):
    return {d: tags_to_spans(p.argmax(-1).tolist(), ENCD[d][1], text_d[d]) for d, p in probs.items()}

def train(model, tag, epochs, lr, head_lr, bs=8, teacher=None, T=2.0, alpha=0.5):
    model.to(DEVICE); w = class_weights(TR_EX).to(DEVICE)
    groups = []
    bbp = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head")]
    if bbp: groups.append(dict(params=bbp, lr=lr))
    groups.append(dict(params=list(model.head.parameters()), lr=head_lr))
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    steps = epochs * math.ceil(len(TR_EX) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    if teacher: teacher.to(DEVICE).eval()
    best_f, best_r, path = -1, None, f"{CK2}/{tag}.pt"; t_train = 0.0
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(epochs):
        model.train(); random.shuffle(TR_EX); t0 = time.time(); run = 0.0
        for i in range(0, len(TR_EX), bs):
            ids, m, y = [t.to(DEVICE) for t in collate(TR_EX[i:i + bs])]
            with torch.autocast(**AMP): lg = model(ids, m)
            loss = F.cross_entropy(lg.float().view(-1, NTAGS), y.view(-1), weight=w, ignore_index=-100)
            if teacher:
                with torch.no_grad(), torch.autocast(**AMP): tl = teacher(ids, m).float()
                msk = (y != -100).view(-1)
                kd = F.kl_div(F.log_softmax(lg.float().view(-1, NTAGS)[msk] / T, -1),
                              F.softmax(tl.view(-1, NTAGS)[msk] / T, -1), reduction="batchmean") * T * T
                loss = alpha * loss + (1 - alpha) * kd
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        t_train += time.time() - t0
        r = strict_macro(probs_to_spans(predict_probs(model, VA_EX, ENC), ENC, texts), gold, val_ids)
        print(f"[{tag}] ep{ep+1}/{epochs} loss {run/math.ceil(len(TR_EX)/bs):.4f} | val strictF1 {r['F1']:.4f} P {r['P']:.3f} R {r['R']:.3f} | {time.time()-t0:.0f}s", flush=True)
        if r["F1"] > best_f: best_f, best_r = r["F1"], r; torch.save(model.state_dict(), path)
    model.load_state_dict(torch.load(path))
    mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0
    return best_f, best_r, t_train, mem

def make_submission(model, tag):
    probs = predict_probs(model, TE_EX, ENC_TEST)
    spans = probs_to_spans(probs, ENC_TEST, test_text)
    pred = {d: [[s, e, l] for s, e, l in spans.get(d, [])] for d in test_text}
    jp, zp = f"{SUB_DIR}/{tag}_prediction.json", f"{SUB_DIR}/{tag}.zip"
    json.dump(pred, open(jp, "w"))
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(jp, arcname="prediction.json")   # at archive root
    validate_zip(zp); return zp

def validate_zip(zp):
    z = zipfile.ZipFile(zp); assert z.namelist() == ["prediction.json"], z.namelist()
    p = json.loads(z.read("prediction.json"))
    assert set(p) == set(test_text), "ID mismatch"
    for d, ents in p.items():
        for s, e, l in ents: assert l in LABELS and 0 <= s < e <= len(test_text[d])
    print("VALID:", zp, "|", len(p), "docs |", sum(len(v) for v in p.values()), "entities")

def run_exp(tag, p1, mode, bidir, epochs, lr, head_lr=1e-3, targets=(), r=8, head="linear",
            teacher=None, keep=None, subset=None, submit=True):
    torch.manual_seed(SEED); torch.cuda.empty_cache()
    model = NERModel(load_phase1(p1, bidir=bidir, keep=keep), head)
    set_mode(model, mode, targets, r); desc = count(model); print(tag, "|", p1, mode, "|", desc)
    global TR_EX
    full = TR_EX
    if subset: TR_EX = TR_EX[:subset]
    f, res, t, mem = train(model, tag, epochs, lr, head_lr, teacher=teacher)
    TR_EX = full
    eb = error_breakdown(probs_to_spans(predict_probs(model, VA_EX, ENC), ENC, texts), gold, val_ids)
    RESULTS.append(dict(tag=tag, p1=p1, mode=mode, valF1=round(f, 4), P=round(res["P"], 3), R=round(res["R"], 3),
                        params=desc, train_min=round(t / 60, 1), gpu_GB=round(mem, 1), errors=eb))
    print(res["per_label"]); print("errors:", eb)
    if submit: make_submission(model, tag)
    return model

In [14]:
_ = run_exp("smoke", "D_efficient", "full", bidir=True, epochs=1, lr=1e-4, subset=64, submit=True)
RESULTS.clear()

smoke | D_efficient full | trainable 85.96M / total 85.96M
[smoke] ep1/1 loss 1.5270 | val strictF1 0.0264 P 0.015 R 0.096 | 4s
{'APP': np.float64(0.0), 'RESP': np.float64(0.0), 'A.COUNSEL': np.float64(0.0), 'R.COUNSEL': np.float64(0.016), 'JUDGE': np.float64(0.0), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.0), 'COURT': np.float64(0.211), 'STAT': np.float64(0.014), 'PREC': np.float64(0.001), 'DATE': np.float64(0.074), 'CASENO': np.float64(0.0)}
errors: {'exact': 115, 'boundary': 587, 'label_confusion': 316, 'false_pos': 3633, 'missed': 44}
VALID: /kaggle/working/p2_submissions/smoke.zip | 35 docs | 26864 entities


In [15]:
mA = run_exp("exp1_full_A", "A_decoder", "full", bidir=True, epochs=5, lr=1e-4)
del mA; torch.cuda.empty_cache()

exp1_full_A | A_decoder full | trainable 109.55M / total 109.55M
[exp1_full_A] ep1/5 loss 0.8379 | val strictF1 0.2199 P 0.169 R 0.328 | 38s
[exp1_full_A] ep2/5 loss 0.1676 | val strictF1 0.2827 P 0.239 R 0.385 | 39s
[exp1_full_A] ep3/5 loss 0.0653 | val strictF1 0.3245 P 0.275 R 0.417 | 41s
[exp1_full_A] ep4/5 loss 0.0268 | val strictF1 0.3519 P 0.308 R 0.441 | 42s
[exp1_full_A] ep5/5 loss 0.0124 | val strictF1 0.3472 P 0.307 R 0.427 | 41s
{'APP': np.float64(0.041), 'RESP': np.float64(0.224), 'A.COUNSEL': np.float64(0.025), 'R.COUNSEL': np.float64(0.341), 'JUDGE': np.float64(0.392), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.158), 'COURT': np.float64(0.917), 'STAT': np.float64(0.62), 'PREC': np.float64(0.221), 'DATE': np.float64(0.92), 'CASENO': np.float64(0.363)}
errors: {'exact': 604, 'boundary': 273, 'label_confusion': 71, 'false_pos': 311, 'missed': 114}
VALID: /kaggle/working/p2_submissions/exp1_full_A.zip | 35 docs | 5238 entities


In [16]:
mB = run_exp("exp2_lorma_B", "B_encdec", "lorma", bidir=True, epochs=8, lr=1e-3, targets={"q", "v"}, r=8)
del mB; torch.cuda.empty_cache()

exp2_lorma_B | B_encdec lorma | trainable 0.17M / total 67.22M


/tmp/ipykernel_49/2808519199.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


[exp2_lorma_B] ep1/8 loss 2.2163 | val strictF1 0.0115 P 0.009 R 0.072 | 15s
[exp2_lorma_B] ep2/8 loss 1.3924 | val strictF1 0.0559 P 0.036 R 0.136 | 15s
[exp2_lorma_B] ep3/8 loss 1.0351 | val strictF1 0.0699 P 0.044 R 0.186 | 15s
[exp2_lorma_B] ep4/8 loss 0.8476 | val strictF1 0.0896 P 0.061 R 0.176 | 15s
[exp2_lorma_B] ep5/8 loss 0.7484 | val strictF1 0.0941 P 0.063 R 0.197 | 15s
[exp2_lorma_B] ep6/8 loss 0.6620 | val strictF1 0.0911 P 0.060 R 0.195 | 15s
[exp2_lorma_B] ep7/8 loss 0.6171 | val strictF1 0.1061 P 0.072 R 0.215 | 15s
[exp2_lorma_B] ep8/8 loss 0.5969 | val strictF1 0.0999 P 0.068 R 0.198 | 15s
{'APP': np.float64(0.009), 'RESP': np.float64(0.0), 'A.COUNSEL': np.float64(0.0), 'R.COUNSEL': np.float64(0.044), 'JUDGE': np.float64(0.027), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.211), 'COURT': np.float64(0.458), 'STAT': np.float64(0.165), 'PREC': np.float64(0.021), 'DATE': np.float64(0.283), 'CASENO': np.float64(0.054)}
errors: {'exact': 286, 'boundary': 553, 'label_confus

In [17]:
mC = run_exp("exp3_lora_C", "C_hybrid", "lora", bidir=False, epochs=8, lr=1e-3, targets={"q", "v", "inp"}, r=8)
del mC; torch.cuda.empty_cache()

exp3_lora_C | C_hybrid lora | trainable 0.36M / total 114.64M
[exp3_lora_C] ep1/8 loss 1.4371 | val strictF1 0.1948 P 0.152 R 0.313 | 39s
[exp3_lora_C] ep2/8 loss 0.3437 | val strictF1 0.2250 P 0.177 R 0.330 | 36s
[exp3_lora_C] ep3/8 loss 0.2292 | val strictF1 0.2673 P 0.218 R 0.382 | 36s
[exp3_lora_C] ep4/8 loss 0.1523 | val strictF1 0.2415 P 0.188 R 0.370 | 36s
[exp3_lora_C] ep5/8 loss 0.1028 | val strictF1 0.2708 P 0.220 R 0.376 | 36s
[exp3_lora_C] ep6/8 loss 0.0718 | val strictF1 0.2656 P 0.219 R 0.356 | 36s
[exp3_lora_C] ep7/8 loss 0.0577 | val strictF1 0.2655 P 0.220 R 0.352 | 36s
[exp3_lora_C] ep8/8 loss 0.0522 | val strictF1 0.2606 P 0.217 R 0.342 | 36s
{'APP': np.float64(0.104), 'RESP': np.float64(0.092), 'A.COUNSEL': np.float64(0.094), 'R.COUNSEL': np.float64(0.383), 'JUDGE': np.float64(0.192), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.163), 'COURT': np.float64(0.805), 'STAT': np.float64(0.361), 'PREC': np.float64(0.151), 'DATE': np.float64(0.723), 'CASENO': np.float64(0.1

In [18]:
teacher = NERModel(load_phase1("A_decoder", bidir=True)); teacher.load_state_dict(torch.load(f"{CK2}/exp1_full_A.pt")); teacher.eval()
mD = run_exp("exp4_kd_D", "D_efficient", "full", bidir=True, epochs=6, lr=1e-4,
             keep=[1, 3, 5, 7, 9, 11], teacher=teacher)
del mD, teacher; torch.cuda.empty_cache()

exp4_kd_D | D_efficient full | trainable 55.28M / total 55.28M


/tmp/ipykernel_49/2808519199.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


[exp4_kd_D] ep1/6 loss 1.7793 | val strictF1 0.1627 P 0.127 R 0.248 | 30s
[exp4_kd_D] ep2/6 loss 0.3761 | val strictF1 0.2237 P 0.188 R 0.307 | 30s
[exp4_kd_D] ep3/6 loss 0.1622 | val strictF1 0.2749 P 0.242 R 0.340 | 29s
[exp4_kd_D] ep4/6 loss 0.0805 | val strictF1 0.2952 P 0.267 R 0.346 | 30s
[exp4_kd_D] ep5/6 loss 0.0462 | val strictF1 0.3162 P 0.298 R 0.359 | 29s
[exp4_kd_D] ep6/6 loss 0.0341 | val strictF1 0.3159 P 0.294 R 0.363 | 29s
{'APP': np.float64(0.095), 'RESP': np.float64(0.202), 'A.COUNSEL': np.float64(0.08), 'R.COUNSEL': np.float64(0.308), 'JUDGE': np.float64(0.351), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.128), 'COURT': np.float64(0.905), 'STAT': np.float64(0.392), 'PREC': np.float64(0.167), 'DATE': np.float64(0.853), 'CASENO': np.float64(0.312)}
errors: {'exact': 495, 'boundary': 317, 'label_confusion': 89, 'false_pos': 164, 'missed': 161}
VALID: /kaggle/working/p2_submissions/exp4_kd_D.zip | 35 docs | 4774 entities


In [19]:
import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != "errors"} for r in RESULTS]); print(df.to_string(index=False))
for r in RESULTS: print(r["tag"], r["errors"])

         tag          p1  mode  valF1     P     R                            params  train_min  gpu_GB
 exp1_full_A   A_decoder  full 0.3519 0.308 0.441 trainable 109.55M / total 109.55M        3.3     4.9
exp2_lorma_B    B_encdec lorma 0.1061 0.072 0.215    trainable 0.17M / total 67.22M        1.9     1.8
 exp3_lora_C    C_hybrid  lora 0.2708 0.220 0.376   trainable 0.36M / total 114.64M        4.7     3.8
   exp4_kd_D D_efficient  full 0.3162 0.298 0.359   trainable 55.28M / total 55.28M        2.9     3.4
exp1_full_A {'exact': 604, 'boundary': 273, 'label_confusion': 71, 'false_pos': 311, 'missed': 114}
exp2_lorma_B {'exact': 286, 'boundary': 553, 'label_confusion': 167, 'false_pos': 1814, 'missed': 56}
exp3_lora_C {'exact': 499, 'boundary': 403, 'label_confusion': 103, 'false_pos': 658, 'missed': 57}
exp4_kd_D {'exact': 495, 'boundary': 317, 'label_confusion': 89, 'false_pos': 164, 'missed': 161}


In [20]:
for zp in sorted(glob.glob(f"{SUB_DIR}/*.zip")): validate_zip(zp)

VALID: /kaggle/working/p2_submissions/exp1_full_A.zip | 35 docs | 5238 entities
VALID: /kaggle/working/p2_submissions/exp2_lorma_B.zip | 35 docs | 14649 entities
VALID: /kaggle/working/p2_submissions/exp3_lora_C.zip | 35 docs | 7798 entities
VALID: /kaggle/working/p2_submissions/exp4_kd_D.zip | 35 docs | 4774 entities
VALID: /kaggle/working/p2_submissions/smoke.zip | 35 docs | 26864 entities


In [21]:
def rebuild(maxlen, overlap, neg_keep):
    global TR_EX, VA_EX, TE_EX, windows
    def windows(n, L=None, ov=None):
        if n <= maxlen: return [(0, n)]
        step, out, i = maxlen - overlap, [], 0
        while i + maxlen < n: out.append((i, i + maxlen)); i += step
        out.append((n - maxlen, n)); return out
    TR_EX = build_examples(train_ids, ENC, True, neg_keep)
    VA_EX = build_examples(val_ids, ENC, True, 1.0)
    TE_EX = build_examples(sorted(test_text), ENC_TEST, False)
    print(f"windows maxlen={maxlen}: train/val/test", len(TR_EX), len(VA_EX), len(TE_EX))

CFGS = {
  "A_512_lr5e-5":    dict(maxlen=512,  overlap=128, neg=0.3, lr=5e-5, bidir=True),
  "A_1024_lr1e-4":   dict(maxlen=1024, overlap=256, neg=0.3, lr=1e-4, bidir=True),
  "A_1024_lr5e-5":   dict(maxlen=1024, overlap=256, neg=0.3, lr=5e-5, bidir=True),
  "A_512_causal":    dict(maxlen=512,  overlap=128, neg=0.3, lr=1e-4, bidir=False),
}
for tag, c in CFGS.items():
    rebuild(c["maxlen"], c["overlap"], c["neg"])
    m = run_exp("abl_" + tag, "A_decoder", "full", bidir=c["bidir"], epochs=5, lr=c["lr"], submit=False)
    del m; torch.cuda.empty_cache()

import pandas as pd
print(pd.DataFrame([{k: v for k, v in r.items() if k != "errors"} for r in RESULTS]).sort_values("valF1", ascending=False).to_string(index=False))

windows maxlen=512: train/val/test 1069 108 648
abl_A_512_lr5e-5 | A_decoder full | trainable 109.55M / total 109.55M
[abl_A_512_lr5e-5] ep1/5 loss 0.9143 | val strictF1 0.2645 P 0.206 R 0.389 | 42s
[abl_A_512_lr5e-5] ep2/5 loss 0.1546 | val strictF1 0.3227 P 0.265 R 0.446 | 42s
[abl_A_512_lr5e-5] ep3/5 loss 0.0615 | val strictF1 0.3610 P 0.326 R 0.429 | 42s
[abl_A_512_lr5e-5] ep4/5 loss 0.0242 | val strictF1 0.3514 P 0.317 R 0.423 | 42s
[abl_A_512_lr5e-5] ep5/5 loss 0.0132 | val strictF1 0.3481 P 0.319 R 0.406 | 42s
{'APP': np.float64(0.141), 'RESP': np.float64(0.215), 'A.COUNSEL': np.float64(0.16), 'R.COUNSEL': np.float64(0.274), 'JUDGE': np.float64(0.31), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.28), 'COURT': np.float64(0.931), 'STAT': np.float64(0.571), 'PREC': np.float64(0.192), 'DATE': np.float64(0.846), 'CASENO': np.float64(0.413)}
errors: {'exact': 583, 'boundary': 310, 'label_confusion': 56, 'false_pos': 246, 'missed': 113}
windows maxlen=1024: train/val/test 555 54 326
ab

In [22]:
from collections import Counter
model = NERModel(load_phase1("A_decoder", bidir=True))
model.load_state_dict(torch.load(f"{CK2}/abl_A_512_lr5e-5.pt", map_location="cpu")); model.to(DEVICE)
pred = probs_to_spans(predict_probs(model, VA_EX, ENC), ENC, texts)

shown = 0
for d in val_ids:
    P = set(pred.get(d, []))
    for g in sorted(gold[d]):
        if g in P: continue
        ov = [p for p in P if p[0] < g[1] and g[0] < p[1] and p[2] == g[2]]
        if ov and shown < 30:
            p = ov[0]; shown += 1
            print(g[2], "| GOLD:", repr(texts[d][g[0]:g[1]]), "| PRED:", repr(texts[d][p[0]:p[1]]))

print("train label counts:", dict(Counter(l for d in train_ids for _, _, l in gold[d])))
print("val label counts:  ", dict(Counter(l for d in val_ids for _, _, l in gold[d])))

CASENO | GOLD: 'O.M.P. (COMM) 399/2019' | PRED: '.M.P. (COMM) 399/2019 &'
CASENO | GOLD: 'IA No.13333/2019 (stay)' | PRED: 'IA No.13333/2019'
A.COUNSEL | GOLD: 'Pinky Anand' | PRED: 'y'
A.COUNSEL | GOLD: 'Hemant Arya' | PRED: 'Hemant'
A.COUNSEL | GOLD: 'Kirti Dua' | PRED: 'Du'
A.COUNSEL | GOLD: 'Rinky' | PRED: 'y'
A.COUNSEL | GOLD: 'Sharvan Kumar' | PRED: 'van'
RESP | GOLD: 'BHARAT BIOTECH INTERNATIONAL LTD' | PRED: 'BHARAT BIOTECH INTERNATIONAL LTD.'
R.COUNSEL | GOLD: 'Karthik Jayashankar' | PRED: 'thik Jayashankar'
R.COUNSEL | GOLD: 'Sughosh Subramanyam Neergundh' | PRED: 'ram'
CASENO | GOLD: 'O.M.P. (COMM) 407/2019' | PRED: '407/2019 &'
CASENO | GOLD: 'IA Nos.13543/2019 (stay)' | PRED: 'IA Nos.13543/2019'
A.COUNSEL | GOLD: 'Pinky Anand' | PRED: 'y'
A.COUNSEL | GOLD: 'Bhagwan Swarup Shukla' | PRED: 'wan Swarup'
A.COUNSEL | GOLD: 'Hemant Arya' | PRED: 'Hemant'
A.COUNSEL | GOLD: 'Kirti Dua' | PRED: 'Du'
A.COUNSEL | GOLD: 'Rinky' | PRED: 'y'
A.COUNSEL | GOLD: 'Sharvan Kumar' | PRED: 'va

In [23]:
rebuild(512, 128, 0.3)

mB2 = run_exp("exp2b_lorma_B_wide", "B_encdec", "lorma", bidir=True, epochs=12, lr=1e-3,
              targets={"q", "k", "v", "proj", "w1", "w2", "w3"}, r=16)
del mB2; torch.cuda.empty_cache()

mC2 = run_exp("exp3b_lora_C_wide", "C_hybrid", "lora", bidir=False, epochs=12, lr=1e-3,
              targets={"q", "k", "v", "proj", "inp", "out", "w1", "w2", "w3"}, r=16)
del mC2; torch.cuda.empty_cache()

windows maxlen=512: train/val/test 1069 108 648
exp2b_lorma_B_wide | B_encdec lorma | trainable 1.54M / total 68.60M


/tmp/ipykernel_49/2808519199.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


[exp2b_lorma_B_wide] ep1/12 loss 2.2315 | val strictF1 0.0169 P 0.010 R 0.067 | 18s
[exp2b_lorma_B_wide] ep2/12 loss 1.2962 | val strictF1 0.0771 P 0.056 R 0.135 | 19s
[exp2b_lorma_B_wide] ep3/12 loss 0.9834 | val strictF1 0.0817 P 0.055 R 0.197 | 18s
[exp2b_lorma_B_wide] ep4/12 loss 0.8493 | val strictF1 0.0896 P 0.062 R 0.210 | 18s
[exp2b_lorma_B_wide] ep5/12 loss 0.7512 | val strictF1 0.1452 P 0.107 R 0.256 | 18s
[exp2b_lorma_B_wide] ep6/12 loss 0.6676 | val strictF1 0.1313 P 0.093 R 0.243 | 18s
[exp2b_lorma_B_wide] ep7/12 loss 0.5787 | val strictF1 0.1684 P 0.128 R 0.260 | 18s
[exp2b_lorma_B_wide] ep8/12 loss 0.5256 | val strictF1 0.1642 P 0.124 R 0.268 | 18s
[exp2b_lorma_B_wide] ep9/12 loss 0.4800 | val strictF1 0.1582 P 0.119 R 0.264 | 18s
[exp2b_lorma_B_wide] ep10/12 loss 0.4313 | val strictF1 0.1888 P 0.145 R 0.294 | 18s
[exp2b_lorma_B_wide] ep11/12 loss 0.4066 | val strictF1 0.1871 P 0.143 R 0.296 | 18s
[exp2b_lorma_B_wide] ep12/12 loss 0.3936 | val strictF1 0.1842 P 0.141 R 0

In [24]:
def get_probs(ckpt, maxlen, overlap):
    rebuild(maxlen, overlap, 0.3)
    m = NERModel(load_phase1("A_decoder", bidir=True))
    m.load_state_dict(torch.load(ckpt, map_location="cpu")); m.to(DEVICE).eval()
    pv = predict_probs(m, VA_EX, ENC); pt = predict_probs(m, TE_EX, ENC_TEST)
    del m; torch.cuda.empty_cache(); return pv, pt

MEMBERS = [(f"{CK2}/exp1_full_A.pt",       512, 128),
           (f"{CK2}/abl_A_512_lr5e-5.pt",  512, 128),
           (f"{CK2}/abl_A_1024_lr1e-4.pt", 1024, 256),
           (f"{CK2}/abl_A_1024_lr5e-5.pt", 1024, 256)]

PV, PT = [], []
for ck, ml, ov in MEMBERS:
    pv, pt = get_probs(ck, ml, ov); PV.append(pv); PT.append(pt)
    f = strict_macro(probs_to_spans(pv, ENC, texts), gold, val_ids)["F1"]
    print(os.path.basename(ck), "val F1", round(f, 4))

def avg(P): return {d: np.mean([p[d] for p in P], 0) for d in P[0]}
ens_val = strict_macro(probs_to_spans(avg(PV), ENC, texts), gold, val_ids)
print("ENSEMBLE val F1", round(ens_val["F1"], 4), "P", round(ens_val["P"], 3), "R", round(ens_val["R"], 3))
print(ens_val["per_label"])

spans = probs_to_spans(avg(PT), ENC_TEST, test_text)
pred = {d: [[s, e, l] for s, e, l in spans.get(d, [])] for d in test_text}
jp, zp = f"{SUB_DIR}/best_ensemble_A_prediction.json", f"{SUB_DIR}/best_ensemble_A.zip"
json.dump(pred, open(jp, "w"))
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(jp, arcname="prediction.json")
validate_zip(zp)
rebuild(512, 128, 0.3)

windows maxlen=512: train/val/test 1069 108 648
exp1_full_A.pt val F1 0.3519
windows maxlen=512: train/val/test 1069 108 648
abl_A_512_lr5e-5.pt val F1 0.361
windows maxlen=1024: train/val/test 555 54 326
abl_A_1024_lr1e-4.pt val F1 0.3435
windows maxlen=1024: train/val/test 555 54 326
abl_A_1024_lr5e-5.pt val F1 0.3199
ENSEMBLE val F1 0.3551 P 0.318 R 0.423
{'APP': np.float64(0.112), 'RESP': np.float64(0.2), 'A.COUNSEL': np.float64(0.084), 'R.COUNSEL': np.float64(0.25), 'JUDGE': np.float64(0.391), 'WIT': np.float64(0.0), 'AUTH': np.float64(0.159), 'COURT': np.float64(0.923), 'STAT': np.float64(0.653), 'PREC': np.float64(0.216), 'DATE': np.float64(0.906), 'CASENO': np.float64(0.366)}
VALID: /kaggle/working/p2_submissions/best_ensemble_A.zip | 35 docs | 5156 entities
windows maxlen=512: train/val/test 1069 108 648


In [25]:
if "_old_set_mode" not in globals():
    _old_set_mode = set_mode
    def set_mode(model, mode, targets=(), r=8):
        if mode == "lorma_norm":
            _old_set_mode(model, "lorma", targets, r)
            for m in model.bb.modules():
                if isinstance(m, RMSNorm):
                    for p in m.parameters(): p.requires_grad = True
            return model
        return _old_set_mode(model, mode, targets, r)

rebuild(512, 128, 0.3)
mB3 = run_exp("exp2c_lorma_B_long", "B_encdec", "lorma_norm", bidir=True, epochs=25, lr=3e-3,
              targets={"q", "k", "v", "proj", "w1", "w2", "w3"}, r=16)
del mB3; torch.cuda.empty_cache()

windows maxlen=512: train/val/test 1069 108 648
exp2c_lorma_B_long | B_encdec lorma_norm | trainable 1.55M / total 68.60M


/tmp/ipykernel_49/2808519199.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


[exp2c_lorma_B_long] ep1/25 loss 2.2251 | val strictF1 0.0239 P 0.015 R 0.067 | 19s
[exp2c_lorma_B_long] ep2/25 loss 1.5419 | val strictF1 0.0233 P 0.014 R 0.066 | 19s
[exp2c_lorma_B_long] ep3/25 loss 1.8591 | val strictF1 0.0014 P 0.001 R 0.004 | 18s
[exp2c_lorma_B_long] ep4/25 loss 1.7968 | val strictF1 0.0003 P 0.000 R 0.001 | 18s
[exp2c_lorma_B_long] ep5/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 18s
[exp2c_lorma_B_long] ep6/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 17s
[exp2c_lorma_B_long] ep7/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 16s
[exp2c_lorma_B_long] ep8/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 16s
[exp2c_lorma_B_long] ep9/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 16s
[exp2c_lorma_B_long] ep10/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 16s
[exp2c_lorma_B_long] ep11/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 16s
[exp2c_lorma_B_long] ep12/25 loss nan | val strictF1 0.0000 P 0.000 R 0.000 | 16s
[exp2c_lorma_

In [26]:
rebuild(512, 128, 0.3)
mB3 = run_exp("exp2d_lorma_B_long", "B_encdec", "lorma", bidir=True, epochs=25, lr=1e-3,
              targets={"q", "k", "v", "proj", "w1", "w2", "w3"}, r=16)
del mB3; torch.cuda.empty_cache()

windows maxlen=512: train/val/test 1069 108 648
exp2d_lorma_B_long | B_encdec lorma | trainable 1.54M / total 68.60M


/tmp/ipykernel_49/2808519199.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); run += loss.item()


[exp2d_lorma_B_long] ep1/25 loss 2.4060 | val strictF1 0.0014 P 0.001 R 0.002 | 18s
[exp2d_lorma_B_long] ep2/25 loss 1.4606 | val strictF1 0.0432 P 0.027 R 0.120 | 19s
[exp2d_lorma_B_long] ep3/25 loss 1.0390 | val strictF1 0.0638 P 0.044 R 0.137 | 19s
[exp2d_lorma_B_long] ep4/25 loss 0.8510 | val strictF1 0.1100 P 0.078 R 0.215 | 18s
[exp2d_lorma_B_long] ep5/25 loss 0.7258 | val strictF1 0.1265 P 0.101 R 0.190 | 18s
[exp2d_lorma_B_long] ep6/25 loss 0.6516 | val strictF1 0.1398 P 0.110 R 0.248 | 18s
[exp2d_lorma_B_long] ep7/25 loss 0.5868 | val strictF1 0.1271 P 0.095 R 0.232 | 18s
[exp2d_lorma_B_long] ep8/25 loss 0.5556 | val strictF1 0.1489 P 0.121 R 0.222 | 18s
[exp2d_lorma_B_long] ep9/25 loss 0.4995 | val strictF1 0.1189 P 0.089 R 0.197 | 18s
[exp2d_lorma_B_long] ep10/25 loss 0.4581 | val strictF1 0.1493 P 0.119 R 0.212 | 18s
[exp2d_lorma_B_long] ep11/25 loss 0.4283 | val strictF1 0.1264 P 0.094 R 0.221 | 18s
[exp2d_lorma_B_long] ep12/25 loss 0.4014 | val strictF1 0.1799 P 0.143 R 0